# Data Construction Notebook

This notebook documents the workflow used to build full_rent_features.csv. Some cells require external geospatial source files, API access, and GIS libraries; the processed CSV is included for the final analysis notebooks.

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import box
from shapely.geometry import Point
import requests
import time

import warnings
warnings.filterwarnings('ignore')

## Residential Parcel Selection

Filter SanGIS parcel and land-use data to the UCSD-adjacent study area.

In [ ]:
xmin, y_min, x_max, y_max = gpd.GeoSeries([box(-117.268, 32.847, -117.181, 32.896)], 
                                          crs=4326).to_crs(2230).total_bounds

parcels = gpd.read_file(
    "data/parcels/Parcels_South.shp",
    bbox=box(xmin, y_min, x_max, y_max)
)
parcels = parcels[parcels.is_valid]

communities = gpd.read_file("data/communities/Community_Points.shp")
study_area = communities[(communities['community'] == 'La Jolla Village')|
    (communities['community'] == 'University City')]

study_area['geometry'] = study_area.buffer(5000)

study_parcels = gpd.sjoin(parcels, study_area, predicate='intersects').drop(columns=['index_right'])

In [ ]:
land_use = gpd.read_file(
    "data/landuse/Land_Use_2024.shp",
    where="DESCRIPTIO = 'Single-Family Detached' OR DESCRIPTIO = 'Single Family Multiple-Units' OR\
    DESCRIPTIO = 'Multi-Family Residential'",
    bbox=box(xmin, y_min, x_max, y_max)
)

study_land = gpd.sjoin(study_parcels, land_use, predicate='intersects')
study_land = study_land[['community', 'DESCRIPTIO', 'geometry']].reset_index(drop=True)
study_land.plot();

In [ ]:
study_land['centroid'] = study_land.geometry.centroid
study_land.head(20)

## Sampling Grid

Create a spatial grid and sampled centroids for collecting rental estimates.

In [ ]:
def create_grid(gdf, cell_size):
    minx, miny, maxx, maxy = gdf.total_bounds

    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)

    polygons = []
    for x in xs:
        for y in ys:
            polygons.append(box(x, y, x + cell_size, y + cell_size))

    grid = gpd.GeoDataFrame({"geometry": polygons}, crs=gdf.crs)
    return grid

grid = create_grid(study_land, 1000)
grid = grid[grid.intersects(study_land.unary_union)]

In [ ]:
ax = study_land.plot()
grid.centroid.plot(ax=ax, color="red", markersize=2)

ax.set_xticks([])
ax.set_yticks([]);

In [ ]:
grid.shape[0]

In [ ]:
grid_sample = grid.sample(49, random_state=10)

grid_sample["centroid"] = grid_sample.geometry.centroid
grid_sample = grid_sample[grid_sample["centroid"].notna()]

grid_sample = grid_sample.to_crs(4326)
grid_sample["lon"] = grid_sample.centroid.x
grid_sample["lat"] = grid_sample.centroid.y

## RentCast Feature Collection

Query rental estimates and summarize rent and rent-per-square-foot features.

In [ ]:
def compute_features(listings):

    rents = []
    psf = []

    if not isinstance(listings, list):
        return {
            "median_rent": None,
            "median_rent_psq": None,
            "listing_count": 0,
        }

    for listing in listings:

        price = listing.get("price")
        sqft = listing.get("squareFootage")

        if price is None or sqft is None:
            continue
            
        if price < 500 or price > 25000:
            continue

        if sqft < 200 or sqft > 10000:
            continue

        listing_psf = price / sqft

        if listing_psf < 0.5 or listing_psf > 15:
            continue

        rents.append(price)
        psf.append(listing_psf)

    return {
        "median_rent": float(np.median(rents)) if rents else None,
        "median_rent_psq": float(np.median(psf)) if psf else None,
        "listing_count": len(rents),
    }
    

API_KEY = "33dd4367252042e489b44b3d76da947d"

headers = {
    "X-Api-Key": API_KEY,
    "accept": "application/json",
}


def query(lat, lon, radius=0.25):
    url = "https://api.rentcast.io/v1/listings/rental/long-term"

    params = {
        "latitude": lat,
        "longitude": lon,
        "radius": radius,
        "limit": 50,
    }

    try: 
        r = requests.get(url, headers=headers, params=params, timeout=15)

        r.raise_for_status()

        listings = r.json()

        return compute_features(listings)
        
    except Exception as e:
        print(f"Error at ({lat}, {lon}): {e}")

        return {
            "median_rent": None,
            "median_rent_psq": None,
            "listing_count": 0
        }

In [ ]:
results_df.to_csv(
    "rent_features.csv",
    index=False
)

In [ ]:
rent_gdf.to_file(
    "rent_features.gpkg",
    driver="GPKG"
)

In [ ]:
results_df['median_rent_psq'].hist(bins=20)

## Additional Rental Samples

Identify unsampled grid cells and collect additional rental feature batches.

In [ ]:
sample1 = pd.read_csv("rent_features.csv")

grid = grid.to_crs(4326)

grid["centroid"] = grid.geometry.centroid
grid["lon"] = grid["centroid"].x
grid["lat"] = grid["centroid"].y

grid["coordinate_id"] = (
    grid["lat"].round(6).astype(str)
    + "_"
    + grid["lon"].round(6).astype(str)
)

sample1["coordinate_id"] = (
    sample1["lat"].round(6).astype(str)
    + "_"
    + sample1["lon"].round(6).astype(str)
)

unsampled = grid[~grid["coordinate_id"].isin(sample1["coordinate_id"])]

In [ ]:
unsampled

In [ ]:
sample2 = unsampled.sample(n=50, random_state=10)

results = []

for _, row in sample2.iterrows():

    lat = row["lat"]
    lon = row["lon"]

    features = query(lat, lon)

    results.append({
        "lat": lat,
        "lon": lon,
        **features
    })
    
    time.sleep(1)

results_df = pd.DataFrame(results)

rent_gdf = gpd.GeoDataFrame(
    results_df,
    geometry=gpd.points_from_xy(
        results_df["lon"],
        results_df["lat"]
    ),
    crs=4326
)

In [ ]:
results_df

In [ ]:
results_df.to_csv(
    "rent_features2.csv",
    index=False
)

rent_gdf.to_file(
    "rent_features2.gpkg",
    driver="GPKG"
)

In [ ]:
unsampled = unsampled[~unsampled["coordinate_id"].isin(sample2["coordinate_id"])]
unsampled.shape[0]

In [ ]:
sample3 = unsampled.sample(n=46, random_state=10)

results = []

for _, row in sample3.iterrows():

    lat = row["lat"]
    lon = row["lon"]

    features = query(lat, lon)

    results.append({
        "lat": lat,
        "lon": lon,
        **features
    })
    
    time.sleep(1)

results_df = pd.DataFrame(results)

rent_gdf = gpd.GeoDataFrame(
    results_df,
    geometry=gpd.points_from_xy(
        results_df["lon"],
        results_df["lat"]
    ),
    crs=4326
)

In [ ]:
results_df

In [ ]:
results_df.to_csv(
    "rent_features3.csv",
    index=False
)

rent_gdf.to_file(
    "rent_features3.gpkg",
    driver="GPKG"
)

## Combine Rental Feature Samples

Merge the rental feature batches and remove incomplete observations.

In [ ]:
data_1 = pd.read_csv("rent_features1.csv")
data_2 = pd.read_csv("rent_features2.csv")
data_3 = pd.read_csv("rent_features3.csv")

full_rent = pd.concat([data_1, data_2, data_3])
full_rent.shape[0]

In [ ]:
full_rent = full_rent.dropna()

In [ ]:
full_rent.to_csv(
    "full_rent_features.csv",
    index=False
)

In [ ]:
full_rent['median_rent_psq'].hist(bins=20);

In [ ]:
full_rent["listing_count"].sum()

## Distance Features

Add distances to major nearby landmarks used as candidate predictors.

In [ ]:
full_rent = pd.read_csv("full_rent.csv")

full_rent_features = gpd.GeoDataFrame(
    full_rent,
    geometry=gpd.points_from_xy(full_rent["lon"], full_rent["lat"]),
    crs=4326
)

In [ ]:
full_rent_features = full_rent_features.to_crs(2230)

features = gpd.GeoSeries(
    {
        "shores_beach": Point(-117.25693, 32.85800), 
        "utc": Point(-117.21086, 32.87095), 
        "downtown": Point(-117.16450, 32.71880)
    },
    crs=4326
).to_crs(2230)

distances = full_rent_features.geometry.apply(lambda x: features.distance(x))

In [ ]:
for column in distances.columns:
    full_rent_features[f"{column}_ft"] = distances[column].round(6)

In [ ]:
full_rent_features

## OpenStreetMap Amenity Features

Use OSMnx/OpenStreetMap features to compute nearest school, park, and bus stop distances.

In [ ]:
import osmnx as osm

tags = {
    "amenity": "school",
    "leisure": "park",
    "highway": "bus_stop"
}

locs = osm.features_from_place("San Diego, California, USA", tags)
locs = locs[locs.geometry.notnull()]
locs["geometry"] = locs["geometry"].centroid
locs = locs.to_crs(2230)

In [ ]:
locs

In [ ]:
schools = locs[locs["amenity"] == "school"]
parks = locs[locs["leisure"] == "park"]
bus_stops = locs[locs["highway"] == "bus_stop"]

full_rent_features["nearest_school_ft"] = full_rent_features.geometry.apply(
    lambda x: schools.distance(x).min()
)

full_rent_features["nearest_park_ft"] = full_rent_features.geometry.apply(
    lambda x: parks.distance(x).min()
)

full_rent_features["nearest_bus_stop_ft"] = full_rent_features.geometry.apply(
    lambda x: schools.distance(x).min()
)

In [ ]:
full_rent_features

## Final Processed Dataset

Export the processed dataset used by the regression notebooks.

In [ ]:
full_rent_features = full_rent_features.drop(columns=["geometry"])

full_rent_features.to_csv(
    "full_rent_features.csv",
    index=False
)

In [ ]:
full_rent_features["listing_count"].sum()